# TRM Compiler Pass Ordering — Real LLVM Training

Train a 60K-param TRM model on **real LLVM** via CompilerGym, benchmark vs LLVM -Oz/-O3.

**Before running:** Go to Runtime → Change runtime type → Select **GPU** (T4 or higher)

In [1]:
# Cell 0: Restart runtime (clears any broken venv from previous runs)
import os

# Delete old venv if it exists
import subprocess
if os.path.exists('/content/trm-env'):
    print("Removing old venv...")
    subprocess.run(['rm', '-rf', '/content/trm-env'], check=True)
    print("Old venv removed.")

print("Now go to Runtime → Restart runtime and run all cells fresh!")
print("IMPORTANT: Don't run this cell again after restart - just run cells 1-4.")

Now go to Runtime → Restart runtime and run all cells fresh!
IMPORTANT: Don't run this cell again after restart - just run cells 1-4.


In [2]:
# Cell 1: Clone repo
import os
import subprocess

PROJECT_DIR = '/content/trm-youtubevids'
VENV_DIR = '/content/trm-env'

if not os.path.exists(PROJECT_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/Cree0618/trm-youtubevids.git', PROJECT_DIR], check=True)
else:
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull'], check=True)

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

Working directory: /content/trm-youtubevids


In [3]:
!pip install requests

In [4]:
import subprocess

result = subprocess.run(
    ["/content/trm-env/bin/python", "-c", "import compiler_gym"],
    capture_output=True,
    text=True,
)
print(result.stderr[:1000])


FileNotFoundError: [Errno 2] No such file or directory: '/content/trm-env/bin/python'

In [6]:
# Cell 2: Create Python 3.11 venv and install dependencies
# compiler_gym requires Python <= 3.11 (uses distutils removed in 3.12)

import subprocess
import os

# Create venv if needed
if not os.path.exists(VENV_DIR):
    result = subprocess.run(['python3.11', '-m', 'venv', VENV_DIR], capture_output=True, text=True)
    print(f"Created venv: {result.returncode == 0}")
else:
    print(f"Venv exists: {VENV_DIR}")

VENV_PIP = f'{VENV_DIR}/bin/pip'
VENV_PYTHON = f'{VENV_DIR}/bin/python'

# Upgrade pip
subprocess.run([VENV_PIP, 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True)

# Install torch (CPU version to avoid CUDA issues)
subprocess.run([VENV_PIP, 'install', 'torch', 'numpy<2.0', '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)

# Install pydantic, protobuf, and other required deps (including gym!)
subprocess.run([VENV_PIP, 'install', 'pydantic', 'protobuf==3.20.3', 'requests', 'docker', 'fasteners', 'absl-py', 'deprecated', 'tabulate', 'gym'], check=True)

# Install grpcio NEWER version (has pre-built wheels for Python 3.11)
# The trick: install newer grpcio FIRST, then install compiler_gym with --no-deps
subprocess.run([VENV_PIP, 'install', 'grpcio'], check=True)

# Install compiler_gym without dependencies
subprocess.run([VENV_PIP, 'install', 'compiler_gym', '--no-deps'], check=True)

# Verify
result = subprocess.run([VENV_PYTHON, '-c', 'import compiler_gym; print(f"CompilerGym: {compiler_gym.__version__}")'], capture_output=True, text=True)
print(result.stdout.strip())
if result.returncode != 0:
    print(f"Error: {result.stderr}")

Venv exists: /content/trm-env

Error: Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/content/trm-env/lib/python3.11/site-packages/compiler_gym/__init__.py", line 37, in <module>
    from compiler_gym.envs import COMPILER_GYM_ENVS, CompilerEnv
  File "/content/trm-env/lib/python3.11/site-packages/compiler_gym/envs/__init__.py", line 6, in <module>
    from compiler_gym.envs.compiler_env import CompilerEnv
  File "/content/trm-env/lib/python3.11/site-packages/co

In [3]:
# Cell 3: Quick smoke test - verify CompilerGym works

import subprocess

VENV_PYTHON = '/content/trm-env/bin/python'

test_code = '''
import compiler_gym
env = compiler_gym.make("llvm-v0", benchmark="cbench-v1/qsort",
    observation_space="Autophase", reward_space="IrInstructionCountOz")
obs = env.reset()
ap = env.observation["Autophase"]
ic = env.observation["IrInstructionCount"]
print(f"Autophase: {len(ap)} features, Initial inst: {ic}")
for i in range(3):
    _, reward, done, _ = env.step(env.action_space.sample())
    inst = env.observation["IrInstructionCount"]
    print(f"  Step {i}: inst={inst} reward={reward:.2f} done={done}")
env.close()
print("CompilerGym OK!")
'''

result = subprocess.run([VENV_PYTHON, '-c', test_code], capture_output=False, text=True)

In [5]:
# Cell 4: Run TRM on REAL LLVM - train and benchmark

import subprocess

VENV_PYTHON = '/content/trm-env/bin/python'
PROJECT_DIR = '/content/trm-youtubevids'

cmd = [
    VENV_PYTHON,
    f'{PROJECT_DIR}/trm_compiler_real_llvm.py',
    '--epochs', '10',
    '--episodes', '10',
    '--benchmarks', 'qsort', 'adpcm',
    '--max-steps', '20',
    '--batch-size', '64',
    '--num-random', '20',
    '--seed', '42'
]

# Note: No --synthetic flag = uses real CompilerGym
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[:500])

[warn] compiler_gym not found, falling back to --synthetic
Device: cpu  |  CompilerGym: False  |  Benchmarks: ['qsort', 'adpcm']

Phase 1: Generating training traces
  400 records from 2 benchmarks in 3.5s

Phase 2: TRM model
  Parameters: 60,328

Phase 3: Training
  epoch   1/10  loss=3.7309  pass=3.5414  entropy=3.5653  0.0s  lr=9.76e-04
  epoch   2/10  loss=3.4440  pass=3.5483  entropy=3.4791  0.0s  lr=9.05e-04
  epoch   3/10  loss=3.3508  pass=3.4512  entropy=3.4846  0.1s  lr=7.94e-04
  epoch   4/10  loss=3.3027  pass=3.4155  entropy=3.4748  0.0s  lr=6.55e-04
  epoch   5/10  loss=3.2777  pass=3.4091  entropy=3.4730  0.0s  lr=5.00e-04
  epoch   6/10  loss=3.2487  pass=3.3625  entropy=3.4720  0.1s  lr=3.45e-04
  epoch   7/10  loss=3.2220  pass=3.3434  entropy=3.4849  0.1s  lr=2.06e-04
  epoch   8/10  loss=3.2176  pass=3.4657  entropy=3.4976  0.1s  lr=9.55e-05
  epoch   9/10  loss=3.2027  pass=3.4261  entropy=3.4934  0.1s  lr=2.45e-05
  epoch  10/10  loss=3.1989  pass=3.2831  entropy=

## Results

The script outputs benchmark tables comparing:
- **LLVM-Oz**: Fixed -Oz pipeline
- **LLVM-O3**: Fixed -O3 pipeline
- **Random(N)**: Best of N random trials
- **TRM-loop**: TRM with closed-loop feedback
- **TRM-blind**: TRM without feedback